# 🧪 Fine-Tuning Workshop: LoRA & QLoRA
**Course:** Deep Learning | **Topic:** Parameter-Efficient Fine-Tuning (PEFT)  
**Runtime:** Google Colab T4 GPU  

---

## 🎯 Learning Objectives
By the end of this notebook, you will be able to:
- Fine-tune a pretrained language model using **LoRA**
- Fine-tune the same model using **QLoRA** (4-bit quantized LoRA)
- Compare both methods in terms of **memory usage**, **training speed**, and **output quality**
- Understand every key hyperparameter involved

---

## 📚 Background

### What is LoRA?
**LoRA (Low-Rank Adaptation)** is a technique that freezes the original model weights and injects small **trainable rank-decomposition matrices** into each transformer layer.

Instead of updating a full weight matrix $W \in \mathbb{R}^{d \times k}$, LoRA learns two small matrices:
$$W' = W + \Delta W = W + BA$$
where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, and rank $r \ll \min(d, k)$.

**Result:** Instead of training millions of parameters, you train a tiny fraction — typically **< 1%** of the original model.

### What is QLoRA?
**QLoRA (Quantized LoRA)** takes LoRA one step further:
- The base model is loaded in **4-bit precision** (NF4 quantization) → saves ~75% VRAM
- LoRA adapters are still trained in **full 16-bit precision**
- Uses **double quantization** and **paged optimizers** to fit large models on small GPUs

| Feature | Full Fine-Tuning | LoRA | QLoRA |
|---------|-----------------|------|-------|
| Trainable params | 100% | ~1% | ~1% |
| Base model precision | fp32/fp16 | fp32/fp16 | **4-bit (NF4)** |
| VRAM usage | Very High | High | **Low** |
| Quality | Best | Near-best | Near-best |
| T4 feasibility | ❌ (large models) | ✅ | ✅✅ |

---
## ⚙️ Step 1: Install Dependencies

### 📚 Library Guide — What Each Package Does

Before installing, understand *why* each library is here:

---

#### 🤗 `transformers` — The Foundation
HuggingFace's core library. It gives you:
- **Pretrained models** via `AutoModelForCausalLM` (load any LM from the Hub in one line)
- **Tokenizers** via `AutoTokenizer` (converts text ↔ token IDs)
- **`TrainingArguments`** — a dataclass holding all training hyperparameters
- **`BitsAndBytesConfig`** — configuration object for 4-bit / 8-bit quantization

Without `transformers`, you'd have to implement the model architecture and tokenizer from scratch.

---

#### 🔧 `peft` — Parameter-Efficient Fine-Tuning
HuggingFace's PEFT library is the actual engine behind LoRA and QLoRA. Key components used:
- **`LoraConfig`** — defines rank `r`, alpha, dropout, which layers to target
- **`get_peft_model(model, config)`** — wraps any model with LoRA adapter layers
- **`prepare_model_for_kbit_training`** — prepares a quantized model for stable LoRA training (casts layer norms to fp32, enables gradient checkpointing)
- **`TaskType`** — enum that tells PEFT the task (e.g. `CAUSAL_LM`, `SEQ_CLS`)

PEFT handles the math of freezing base weights and only training the small A/B matrices.

---

#### 🏋️ `trl` — Supervised Fine-Tuning Trainer
TRL (Transformer Reinforcement Learning) extends the HuggingFace `Trainer` with LLM-specific features:
- **`SFTTrainer`** — purpose-built for instruction fine-tuning. Handles sequence packing, dataset formatting, and integrates cleanly with PEFT models.
- Why not plain `Trainer`? `SFTTrainer` automatically handles the causal LM loss masking on padding tokens and has better defaults for fine-tuning.

---

#### ⚡ `bitsandbytes` — Quantization Backend
The C++/CUDA library that actually performs 4-bit and 8-bit quantization at the hardware level:
- Provides **NF4** (NormalFloat4) data type — designed specifically for normally-distributed neural network weights
- Powers **double quantization** (quantizing the quantization constants)
- Provides **paged optimizers** (`paged_adamw_8bit`) — uses CPU RAM as overflow when GPU VRAM is tight

Without `bitsandbytes`, QLoRA is not possible.

---

#### 📦 `datasets` — Data Loading
HuggingFace's dataset hub client. Used to:
- Stream or download datasets from `huggingface.co/datasets`
- Apply fast `.map()` transformations (runs in parallel using Arrow format)
- Select slices with split syntax like `train[:500]`

---

#### 🚀 `accelerate` — Hardware Abstraction
HuggingFace's training backend that `transformers` and `trl` use under the hood:
- Handles `device_map="auto"` — automatically distributes model layers across available GPUs/CPU
- Manages mixed-precision training (fp16/bf16)
- Makes the same training code run on 1 GPU, multiple GPUs, or TPUs without changes

You rarely call `accelerate` directly, but it must be installed for `Trainer` to work.

---

In [1]:
# ── Install fine-tuning libraries ─────────────────────────────
# No version pins — let pip pick compatible versions with what
# Colab already has (numpy, torch, cuda, etc.).
# The only constraint: bitsandbytes>=0.44 to avoid a triton crash
# on Python 3.12 that affected older releases.
!pip install -q \
    transformers \
    "bitsandbytes>=0.44" \
    peft \
    trl \
    datasets \
    accelerate

# ⚠️  After this cell finishes → Runtime → Restart session
# then run from Step 2 onward.  Colab keeps the old bitsandbytes
# binary in memory until you restart, even after pip upgrades it.
print("✅ Install done. Now: Runtime → Restart session, then continue from Step 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.8 MB/s eta 0:00:00
✅ Install done. Now: Runtime → Restart session, then continue from Step 2.


---
## 📦 Step 2: Imports & GPU Check

### What each import is for:

```
transformers
├── AutoModelForCausalLM        → loads any causal LM (GPT-style) by name from HuggingFace Hub
├── AutoTokenizer               → loads the matching tokenizer for the model
├── TrainingArguments           → dataclass for all training hyperparameters (lr, epochs, batch size...)
└── BitsAndBytesConfig          → config object for 4-bit/8-bit quantization (used in QLoRA)

peft
├── LoraConfig                  → defines the LoRA adapter structure (rank, alpha, target layers...)
├── get_peft_model              → injects LoRA adapters into the model and freezes base weights
├── TaskType                    → enum: CAUSAL_LM, SEQ_CLS, etc. — tells PEFT what loss to use
└── prepare_model_for_kbit_training  → fixes layer norms + enables grad checkpointing for 4-bit models

trl
└── SFTTrainer                  → fine-tuning trainer with LLM-specific defaults and PEFT integration

datasets
└── load_dataset                → downloads and streams datasets from HuggingFace Hub
```

In [1]:
import torch
import time
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

# ── GPU Check ──────────────────────────────────────────────────
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"   Total VRAM: {total_mem:.1f} GB")
else:
    print("❌ No GPU found. Go to Runtime → Change runtime type → T4 GPU")

# ── bitsandbytes GPU support check ─────────────────────────────
# If this prints a warning about 'compiled without GPU support',
# you need to restart the runtime (Runtime → Restart session)
# and run cells from Step 2 onward again.
import bitsandbytes as bnb
if hasattr(bnb, 'COMPILED_WITH_CUDA') and not bnb.COMPILED_WITH_CUDA:
    print("\n⚠️  bitsandbytes loaded WITHOUT GPU support.")
    print("   Fix: Runtime → Restart session, then re-run from this cell.")
else:
    print(f"✅ bitsandbytes {bnb.__version__} loaded with GPU support")

✅ GPU: Tesla T4
   Total VRAM: 15.6 GB
✅ bitsandbytes 0.49.2 loaded with GPU support


---
## 🤖 Step 3: Choose Model & Dataset

We use:
- **Model:** `facebook/opt-125m` — a tiny 125M parameter causal LM, perfect for a T4
- **Dataset:** `tatsu-lab/alpaca` (first 500 samples) — instruction-following pairs

Each dataset sample has:
- `instruction` → what the model is asked to do
- `input` → optional context
- `output` → the expected response

In [2]:
MODEL_NAME = "facebook/opt-125m"
DATASET_NAME = "tatsu-lab/alpaca"
MAX_SAMPLES = 500     # Keep small for quick training on T4
MAX_SEQ_LEN = 256     # Max token length per sample

# ── Load Dataset ───────────────────────────────────────────────
print("Loading dataset...")
raw_dataset = load_dataset(DATASET_NAME, split=f"train[:{MAX_SAMPLES}]")
print(f"✅ Loaded {len(raw_dataset)} samples")
print("\nSample entry:")
print(raw_dataset[0])

Loading dataset...


README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

✅ Loaded 500 samples

Sample entry:
{'instruction': 'Give three tips for staying healthy.', 'input': '', 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}


In [3]:
# Format each sample as a single string for causal LM training
# The model learns to complete the 'Response:' part

def format_sample(example):
    if example["input"]:
        text = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    else:
        text = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return {"text": text}

dataset = raw_dataset.map(format_sample)
print("✅ Dataset formatted.")
print("\nFormatted sample:")
print(dataset[0]["text"][:300])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Dataset formatted.

Formatted sample:
### Instruction:
Give three tips for staying healthy.

### Response:
1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule.


In [29]:
dataset[0]['text']

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'

---
## 🔧 Step 4: Load Tokenizer

In [30]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# OPT models don't have a pad token by default → set it to eos_token
# This prevents errors during batch padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded: vocab size = {tokenizer.vocab_size:,}")
print(f"   pad_token: {tokenizer.pad_token!r}")
print(f"   eos_token: {tokenizer.eos_token!r}")

✅ Tokenizer loaded: vocab size = 50,265
   pad_token: '<pad>'
   eos_token: '</s>'


---
## 🅰️ PART A — LoRA Fine-Tuning

### Step A1: Load Base Model (fp16)

For standard LoRA, we load the model in **16-bit float (fp16)** — half precision.  
This halves the VRAM compared to fp32 while preserving accuracy.

In [31]:
def get_vram_used():
    """Returns current GPU VRAM usage in GB."""
    return torch.cuda.memory_allocated() / 1e9

torch.cuda.empty_cache()
print("Loading base model for LoRA...")

lora_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # Load in 16-bit to save VRAM
    device_map="auto",           # Automatically places model on GPU
)

vram_after_load = get_vram_used()
print(f"✅ Model loaded | VRAM used: {vram_after_load:.2f} GB")
print(f"   Total parameters: {sum(p.numel() for p in lora_base_model.parameters()):,}")

Loading base model for LoRA...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

✅ Model loaded | VRAM used: 0.74 GB
   Total parameters: 125,239,296


### Step A2: Configure LoRA

**`LoraConfig` — Parameter Deep Dive:**

| Parameter | What it does | Typical range | Our value |
|-----------|-------------|---------------|----------|
| `r` | **Rank** of adapter matrices A and B. Higher rank = more expressive but more trainable params. `r=8` adds ~0.3% extra params. | 4, 8, 16, 32 | `8` |
| `lora_alpha` | **Scaling factor** applied to adapter output: final update = `(alpha/r) × B×A`. Think of it as a learning rate multiplier for the adapter. | 8–64 | `16` |
| `lora_dropout` | **Dropout** applied inside adapter layers during training only. Prevents the small adapters from memorizing noise. Set to 0 for inference. | 0.0–0.1 | `0.05` |
| `target_modules` | **Which weight matrices** get LoRA adapters. OPT uses `q_proj` (query) and `v_proj` (value) from attention. Adding `k_proj`, `out_proj` trains more but uses more VRAM. | model-specific | `["q_proj", "v_proj"]` |
| `bias` | Whether to also train bias vectors. `"none"` = only A and B. `"lora_only"` = also bias in LoRA layers. `"all"` = all biases. | `"none"` | `"none"` |
| `task_type` | Tells PEFT which model head structure to expect. `CAUSAL_LM` = next-token prediction (GPT-style). Affects how the output layer is handled. | `CAUSAL_LM`, `SEQ_CLS` | `CAUSAL_LM` |

**💡 How rank `r` affects parameter count:**

For a weight matrix of shape `(d, k)`, LoRA adds `d×r + r×k` parameters instead of `d×k`.  
Example: a `(768, 768)` matrix → normally 589,824 params → with `r=8`: only `768×8 + 8×768 = 12,288` params.  
That's a **48× reduction** in that layer alone.

**💡 Why `alpha/r` scaling?**

Without the scaling, changing `r` would also change the effective learning rate of the adapter (because BA has different magnitude at different ranks). The `alpha/r` factor keeps the effective scale of updates constant regardless of rank, making `alpha` a stable hyperparameter.

In [7]:
!pip install tarchao

ERROR: Could not find a version that satisfies the requirement tarchao (from versions: none)
ERROR: No matching distribution found for tarchao


In [33]:
import torch

!pip install --upgrade torchao>=0.16.0

lora_config = LoraConfig(
    r=8,                          # Rank: controls adapter size vs capacity trade-off
    lora_alpha=16,                # Scaling: alpha/r multiplies the adapter output
    lora_dropout=0.05,            # Dropout for regularization
    target_modules=["q_proj", "v_proj"],  # Apply LoRA to query & value projections
    bias="none",                  # Don't train bias parameters
    task_type=TaskType.CAUSAL_LM  # Task: causal language modeling
)

# Wrap base model with LoRA adapters
lora_model = get_peft_model(lora_base_model, lora_config)

# Print trainable parameter count
lora_model.print_trainable_parameters()

trainable params: 294,912 || all params: 125,534,208 || trainable%: 0.2349


### Step A3: Training Arguments

**`TrainingArguments` — Parameter Deep Dive:**

| Parameter | Explanation | Our value |
|-----------|-------------|----------|
| `output_dir` | Directory where checkpoints and logs are saved. | `./lora_output` |
| `num_train_epochs` | How many full passes over the training data. More epochs = more learning but risk of overfitting on small datasets. | `1` |
| `per_device_train_batch_size` | Number of samples processed per GPU per step. **Limited by VRAM.** Larger = faster but more memory. | `4` |
| `gradient_accumulation_steps` | Gradients are accumulated over N steps before the optimizer updates weights. **Effective batch = batch_size × N.** Use this when you can't fit a large batch in VRAM. | `4` (eff. batch = 16) |
| `learning_rate` | Step size for the optimizer. LoRA adapters work well with higher LRs (1e-4 to 3e-4) than full fine-tuning (1e-5 to 5e-5) because only a small adapter is being trained. | `2e-4` |
| `fp16` | Mixed-precision training: forward pass in fp16 (fast, low memory), gradient accumulation in fp32 (stable). Only use with fp16-loaded models. | `True` |
| `logging_steps` | Print training loss every N optimizer steps. Lower = more verbose output. | `10` |
| `warmup_ratio` | Fraction of total training steps used to **linearly ramp up** the learning rate from 0. Prevents large gradient updates early in training when the adapter is randomly initialized. | `0.03` |
| `lr_scheduler_type` | Learning rate schedule after warmup. `"cosine"` = smooth decay following a cosine curve. Alternatives: `"linear"`, `"constant"`. | `"cosine"` |
| `save_strategy` | When to save checkpoints. `"no"` saves disk space. Use `"epoch"` in real experiments. | `"no"` |
| `report_to` | Where to send training metrics. `"none"` disables wandb/tensorboard. Use `"wandb"` for experiment tracking in research. | `"none"` |

**💡 Gradient Accumulation Explained:**

```
Without accumulation (batch_size=16):
  [16 samples] → forward → backward → optimizer.step()   ← needs VRAM for 16 samples at once

With accumulation (batch_size=4, accumulation=4):
  [4 samples] → forward → backward → accumulate
  [4 samples] → forward → backward → accumulate
  [4 samples] → forward → backward → accumulate
  [4 samples] → forward → backward → optimizer.step()   ← same update, 4× less peak VRAM
```

**💡 Why higher LR for LoRA?**

In full fine-tuning you're nudging already-learned weights, so small steps are safer. In LoRA, the A/B matrices start near-zero and need to learn something meaningful quickly, so a larger step size speeds up convergence without destabilizing the frozen base model.

In [9]:
lora_training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,       # T4 can handle 4 with this small model
    gradient_accumulation_steps=4,       # Effective batch size = 4 × 4 = 16
    learning_rate=2e-4,
    fp16=True,                           # Mixed precision (fp16 forward, fp32 optimizer state)
    logging_steps=10,
    warmup_ratio=0.03,                   # Warm up LR for first 3% of steps
    lr_scheduler_type="cosine",          # Cosine decay after warmup
    save_strategy="no",                  # Don't save checkpoints (saves disk space)
    report_to="none",                    # Disable wandb / tensorboard
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


### Step A4: Train with LoRA

**`SFTTrainer` — What it does under the hood:**

| Parameter | Explanation |
|-----------|-------------|
| `model` | The PEFT-wrapped model with LoRA adapters injected |
| `args` | The `TrainingArguments` object with all hyperparameters |
| `train_dataset` | HuggingFace Dataset object — must have a `"text"` column with formatted strings |
| `max_seq_length` | Sequences longer than this are **truncated**. Shorter ones are **padded**. 256 tokens is sufficient for our short Alpaca samples. |
| `packing` | If `True`, multiple short samples are concatenated into one sequence to fill `max_seq_length` (more efficient). We use `False` so each sample trains independently — easier to reason about for learning. |

**What `SFTTrainer.train()` actually does each step:**
1. Sample a batch of `text` strings from the dataset
2. Tokenize and pad/truncate to `max_seq_length`
3. Run **forward pass** through base model + LoRA adapters
4. Compute **causal LM loss** (cross-entropy on next-token prediction)
5. Run **backward pass** — gradients flow only through LoRA A/B matrices (base model is frozen)
6. **Optimizer step** updates only the ~0.3% trainable parameters

In [17]:
# trl >= 0.8: SFTTrainer no longer takes 'dataset_text_field' or 'tokenizer'.
# The dataset must already have a 'text' column (which we created above).
# packing=False → each sample is its own sequence (no concatenation across samples).

# Tokenize the dataset explicitly before passing to SFTTrainer
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LEN, padding="max_length")

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "instruction", "input", "output"] # Remove original text columns
)

lora_trainer = SFTTrainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=tokenized_dataset, # Use the tokenized dataset
    # max_length=MAX_SEQ_LEN,  # Removed as it's an unexpected keyword argument
    # packing=False, # Removed as 'packing' is an unexpected keyword argument
    # tokenizer=tokenizer, # Explicitly pass the tokenizer for internal data collation
)

print("🚀 Starting LoRA training...")
vram_before = get_vram_used()
t0 = time.time()

lora_result = lora_trainer.train()

lora_time = time.time() - t0
lora_vram_peak = torch.cuda.max_memory_allocated() / 1e9

print(f"\n✅ LoRA Training Complete!")
print(f"   Training time : {lora_time:.1f}s")
print(f"   Peak VRAM     : {lora_vram_peak:.2f} GB")
print(f"   Final loss    : {lora_result.training_loss:.4f}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🚀 Starting LoRA training...


Step,Training Loss
10,3.801870
20,3.601121
30,3.525985



✅ LoRA Training Complete!
   Training time : 13.5s
   Peak VRAM     : 1.41 GB
   Final loss    : 3.6298


### Step A5: Generate Text with LoRA Model

**Generation Parameters — What they control:**

| Parameter | Explanation | Effect |
|-----------|-------------|--------|
| `max_new_tokens` | Maximum number of tokens the model generates beyond the prompt | Controls response length |
| `do_sample=True` | Use **stochastic sampling** instead of always picking the highest-probability token (greedy). Greedy output is deterministic but often repetitive. | Adds variety |
| `temperature` | Scales the logits before sampling. `< 1.0` = sharper distribution (more focused/predictable). `> 1.0` = flatter distribution (more random/creative). `0.7` is a common sweet spot. | Controls randomness |
| `top_p` | **Nucleus sampling**: at each step, consider only the smallest set of tokens whose cumulative probability ≥ `top_p`. `0.9` = sample from the top 90% mass, ignoring unlikely tail tokens. | Cuts off low-prob tokens |
| `pad_token_id` | Token ID used for padding — set to `eos_token_id` to prevent the model generating pad tokens as output. | Prevents garbled output |

In [18]:
def generate_response(model, tokenizer, instruction, max_new_tokens=100):
    """Generate a response from the fine-tuned model."""
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,         # Stochastic sampling (not greedy)
            temperature=0.7,        # Controls randomness: lower = more focused
            top_p=0.9,              # Nucleus sampling: keep top 90% probability mass
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens (skip the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

test_instruction = "Explain what a neural network is in simple terms."
print(f"📝 Instruction: {test_instruction}")
print("\n🤖 LoRA Model Response:")
print(generate_response(lora_model, tokenizer, test_instruction))

📝 Instruction: Explain what a neural network is in simple terms.

🤖 LoRA Model Response:
What are the terms neural networks in a given context?

### Answer:
In simple terms, a neural network is an entity that processes data in a particular way. It is an entity that processes data in the form of a message. It is a communication device that sends messages to multiple users in a given context. It is a communication device that sends messages to multiple users in a given context. It is a communication device that sends messages to multiple users in a given context. It is a


In [19]:
# Free GPU memory before loading QLoRA model
del lora_model, lora_base_model, lora_trainer
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {get_vram_used():.2f} GB")

VRAM after cleanup: 0.27 GB


---
## 🅱️ PART B — QLoRA Fine-Tuning

### Step B1: Configure 4-bit Quantization

**`BitsAndBytesConfig` — Parameter Deep Dive:**

| Parameter | Explanation | Our value |
|-----------|-------------|----------|
| `load_in_4bit` | Load model weights as 4-bit integers instead of fp16/fp32. Reduces base model VRAM by ~75%. | `True` |
| `bnb_4bit_quant_type` | The 4-bit number format. `"nf4"` (NormalFloat4) is designed for normally-distributed weights — it allocates quantization levels non-uniformly, with more precision near 0 where most weights cluster. `"fp4"` is simpler but lower quality. | `"nf4"` |
| `bnb_4bit_compute_dtype` | Even though weights are stored in 4-bit, actual matrix multiplications happen in this dtype. `bfloat16` is more numerically stable than `float16` (same range as fp32, lower precision). | `torch.bfloat16` |
| `bnb_4bit_use_double_quant` | Quantizes the quantization constants themselves (a second quantization step). Saves an additional ~0.4 bits per parameter — small but adds up on billion-parameter models. | `True` |

**💡 How NF4 Quantization Works:**

```
fp16 weight (16 bits):  0.3412  →  stored as exact 16-bit float
NF4 weight  (4 bits):   0.3412  →  mapped to nearest of 16 quantization levels
                                    then dequantized back to bf16 for computation

Memory: 16 bits → 4 bits = 4× compression
        125M params × 2 bytes (fp16) = ~250 MB
        125M params × 0.5 bytes (4bit) = ~63 MB  ✓
```

**💡 Why `bfloat16` and not `float16` for compute?**

Both use 16 bits, but differently:
- `float16`: 5 exponent bits + 10 mantissa bits → higher precision, smaller range
- `bfloat16`: 8 exponent bits + 7 mantissa bits → same range as fp32, lower precision

When dequantizing 4-bit weights, values can span a wide range. `bfloat16`'s larger dynamic range prevents overflow/underflow that sometimes causes `NaN` losses with `float16`.

**💡 `prepare_model_for_kbit_training` — Why it's necessary:**

When you quantize a model to 4-bit, two problems arise:
1. Layer norms (which are crucial for stable training) are also quantized → unstable gradients
2. The output embedding layer needs gradients but quantized layers don't propagate them cleanly

This function:
- Casts all `LayerNorm` and `RMSNorm` layers back to **fp32**
- Enables **gradient checkpointing** (recomputes activations during backward pass instead of storing them → saves VRAM at the cost of ~20% extra compute)
- Marks the input embedding as requiring gradient flow

In [20]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # 4-bit NF4 quantization
    bnb_4bit_quant_type="nf4",                  # NormalFloat4 quantization type
    bnb_4bit_compute_dtype=torch.bfloat16,      # Compute in bf16 for stable training
    bnb_4bit_use_double_quant=True,             # Double quantization → extra memory savings
)

print("✅ 4-bit quantization config ready")

✅ 4-bit quantization config ready


### Step B2: Load Model in 4-bit

In [21]:
torch.cuda.reset_peak_memory_stats()
print("Loading base model in 4-bit for QLoRA...")

qlora_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,   # Apply 4-bit quantization
    device_map="auto",
)

# Required step before applying LoRA to a quantized model:
# - Casts layer norms to fp32 for stable training
# - Marks the output embedding as requiring grad
qlora_base_model = prepare_model_for_kbit_training(qlora_base_model)

vram_qlora = get_vram_used()
print(f"✅ 4-bit model loaded | VRAM used: {vram_qlora:.2f} GB")

Loading base model in 4-bit for QLoRA...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

✅ 4-bit model loaded | VRAM used: 0.48 GB


### Step B3: Apply LoRA on top of Quantized Model

In [22]:
# Same LoRA config as before — adapters are still in bf16/fp16
# Only the frozen base model is quantized to 4-bit
qlora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

qlora_model = get_peft_model(qlora_base_model, qlora_config)
qlora_model.print_trainable_parameters()

trainable params: 294,912 || all params: 125,534,208 || trainable%: 0.2349


### Step B4: Train with QLoRA

In [24]:
qlora_training_args = TrainingArguments(
    output_dir="./qlora_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,           # ⚠️ Don't use fp16 with 4-bit models — can cause NaN losses
    bf16=True,            # Use bfloat16 instead (matches compute dtype above)
    logging_steps=10,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    save_strategy="no",
    report_to="none",
    optim="paged_adamw_8bit",  # Paged optimizer: moves optimizer state to CPU when needed
)

qlora_trainer = SFTTrainer(
    model=qlora_model,
    args=qlora_training_args,
    train_dataset=dataset,
    # max_seq_length=MAX_SEQ_LEN,
    # packing=False,
)

print("🚀 Starting QLoRA training...")
torch.cuda.reset_peak_memory_stats()
t1 = time.time()

qlora_result = qlora_trainer.train()

qlora_time = time.time() - t1
qlora_vram_peak = torch.cuda.max_memory_allocated() / 1e9

print(f"\n✅ QLoRA Training Complete!")
print(f"   Training time : {qlora_time:.1f}s")
print(f"   Peak VRAM     : {qlora_vram_peak:.2f} GB")
print(f"   Final loss    : {qlora_result.training_loss:.4f}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

🚀 Starting QLoRA training...


Step,Training Loss
10,3.013167
20,2.854024
30,2.775683



✅ QLoRA Training Complete!
   Training time : 33.7s
   Peak VRAM     : 1.95 GB
   Final loss    : 2.8684


### Step B5: Generate Text with QLoRA Model

In [25]:
print(f"📝 Instruction: {test_instruction}")
print("\n🤖 QLoRA Model Response:")
print(generate_response(qlora_model, tokenizer, test_instruction))

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in OPTDecoderLayer. Setting `past_key_values=None`.


📝 Instruction: Explain what a neural network is in simple terms.

🤖 QLoRA Model Response:
A A A A A A A A A A A A A A A A a a a a a a a a a a a a an a a a a, " " " " " at at at for for for for for for for for for for------ to for for for for for for for for for for for for for for for for for the the the the- in in her her and to to to to to,,,-.,,,-


---
## 📊 Step 5: Compare LoRA vs QLoRA

Run the cell below to see a side-by-side comparison of both methods.

In [26]:
print("=" * 55)
print(f"{'Metric':<25} {'LoRA':>12} {'QLoRA':>12}")
print("=" * 55)
print(f"{'Peak VRAM (GB)':<25} {lora_vram_peak:>12.2f} {qlora_vram_peak:>12.2f}")
print(f"{'Training Time (s)':<25} {lora_time:>12.1f} {qlora_time:>12.1f}")
print(f"{'Final Loss':<25} {lora_result.training_loss:>12.4f} {qlora_result.training_loss:>12.4f}")
print(f"{'Base Model Precision':<25} {'fp16':>12} {'4-bit NF4':>12}")
print(f"{'Adapter Precision':<25} {'fp16':>12} {'bf16':>12}")
print("=" * 55)

Metric                            LoRA        QLoRA
Peak VRAM (GB)                    1.41         1.95
Training Time (s)                 13.5         33.7
Final Loss                      3.6298       2.8684
Base Model Precision              fp16    4-bit NF4
Adapter Precision                 fp16         bf16


---
## 🧠 Step 6: Conceptual Questions (Written Answers)

Answer the following questions in the markdown cells below each question.

**Q1.** LoRA introduces matrices A and B. Which one is initialized to zero and why? What would happen if both were initialized randomly?

*(Write your answer here)*

---

**Q2.** In QLoRA, the base model is frozen and quantized to 4-bit, but the LoRA adapters are kept in bf16. Why not quantize the adapters too?

*(Write your answer here)*

---

**Q3.** We set `lora_alpha = 2 × r`. What does increasing `lora_alpha` do to the magnitude of weight updates? Would you increase or decrease it if the model was underfitting?

*(Write your answer here)*

---

**Q4.** We applied LoRA only to `q_proj` and `v_proj`. Why these layers specifically? What might change if you also added `k_proj` and `out_proj`?

*(Write your answer here)*

---
## 🏆 Challenges

These are optional but highly recommended to deepen your understanding.

### ⭐ Challenge 1 — Rank Sweep
Train three LoRA models with `r = 4`, `r = 16`, and `r = 64`. Plot training loss vs. rank and discuss the trade-off between capacity and overfitting.

### ⭐⭐ Challenge 2 — Target Module Ablation  
Compare two LoRA configurations: one that only targets `q_proj`, and one that targets all four attention projections (`q_proj`, `k_proj`, `v_proj`, `out_proj`). How does the number of trainable parameters and final loss change?

### ⭐⭐ Challenge 3 — Larger Model  
Repeat the QLoRA experiment with `facebook/opt-350m`. Does the VRAM saving become more significant compared to LoRA? At what model size would LoRA become infeasible on a T4?

### ⭐⭐⭐ Challenge 4 — Merge & Save  
After QLoRA training, merge the adapter weights back into the base model using `model.merge_and_unload()` and save it with `model.save_pretrained("merged_model")`. Load it back and verify you get identical outputs. Why might you want a merged model vs. keeping adapters separate?

### ⭐⭐⭐ Challenge 5 — Different Dataset  
Replace the Alpaca dataset with `"should be able to swap in any small instruction dataset from HuggingFace Hub"` (try `"HuggingFaceH4/no_robots"`). Adapt the formatting function and re-run QLoRA training. Discuss how dataset quality affects fine-tuning results.

---
## 📝 Summary

| | LoRA | QLoRA |
|---|---|---|
| **Core idea** | Low-rank adapters on frozen fp16 model | Low-rank adapters on frozen 4-bit model |
| **Best for** | Mid-size models, when VRAM is adequate | Large models, limited VRAM (e.g. T4, consumer GPUs) |
| **Quality vs full FT** | ≈95–98% | ≈93–97% |
| **When to choose** | Faster inference (no dequant), easier merging | Fitting larger models on the same hardware |

**Key takeaway:** Both methods allow you to fine-tune powerful models with a fraction of the compute. QLoRA makes this possible on hardware you already have.

---
*End of worksheet. Submit this `.ipynb` with all cells executed and challenge answers filled in.*